# Calibration — interactive companion

Companion to [Post 4c: Calibration and uncertainty](../posts/04c-calibration.qmd).

Calibration asks: when the model says "90% sure", is it right 90% of the time?
This notebook lets you draw reliability diagrams, watch temperature scaling
snap an overconfident curve onto the diagonal, confirm the subtle fact that
recalibration leaves *ranking* untouched, and see why a calibrated threshold
is what makes confidence-gated agency work.

**You'll do (~20 minutes):**
1. Draw reliability diagrams for over/under/calibrated models.
2. Fit a temperature on held-out data and watch ECE collapse.
3. Confirm calibration ≠ discrimination (risk–coverage unchanged).
4. Gate decisions on confidence and see calibration earn its keep.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 6)
plt.rcParams["figure.dpi"] = 110

from nano_agents.calibration import (
    CalibrationDataset, reliability_curve, ece, nll, brier_score,
    temperature_scale, apply_temperature, gated_decision_accuracy,
)
from nano_agents.calibration.model import sigmoid

## 1. The reliability diagram

A model is calibrated if predictions made with confidence p are right a
fraction p of the time — the reliability curve lies on the diagonal.
`sharpness` β controls it: β=1 calibrated, β>1 overconfident, β<1 timid.

In [ ]:
def plot_reliability(betas):
    plt.plot([0, 1], [0, 1], "--", color="#888", label="perfect")
    colors = {1.0: "#55a467", 2.5: "#c44e52", 0.5: "#3a7ebf"}
    for b in betas:
        ds = CalibrationDataset(n_items=10000, sharpness=b, seed=0)
        conf, acc, _ = reliability_curve(ds.confidence, ds.correct, n_bins=12)
        plt.plot(conf, acc, "o-", color=colors.get(b, "#dd8452"),
                 label=f"β={b}  ECE={ece(ds.confidence, ds.correct):.3f}")
    plt.xlabel("reported confidence"); plt.ylabel("observed accuracy")
    plt.legend(); plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.show()

plot_reliability([1.0, 2.5, 0.5])

### Try this
- Add `4.0` to the list — severe overconfidence sags far below the diagonal.
- Confidence histograms reveal the signature: overconfident models pile mass
  near 1.0.

In [ ]:
for b, c in [(1.0, "#55a467"), (2.5, "#c44e52")]:
    ds = CalibrationDataset(n_items=10000, sharpness=b, seed=0)
    plt.hist(ds.confidence, bins=30, range=(0,1), alpha=0.5, color=c,
             label=f"β={b}: acc={ds.accuracy:.2f}, mean conf={ds.confidence.mean():.2f}")
plt.xlabel("confidence"); plt.ylabel("count"); plt.legend(); plt.show()

## 2. Temperature scaling

Fit a single temperature T on held-out data (minimize NLL) and divide the
logits by it. T>1 softens overconfidence. The fitted T lands near the true β.

In [ ]:
beta = 2.5
ds = CalibrationDataset(n_items=16000, sharpness=beta, seed=0)
(cal_logit, cal_y), (test_logit, test_y) = ds.split(frac=0.5, seed=1)

T = temperature_scale(cal_logit, cal_y)
raw, fixed = sigmoid(test_logit), apply_temperature(test_logit, T)
print(f"fitted T = {T:.2f}  (true beta = {beta})")
print(f"ECE: {ece(raw, test_y):.3f} -> {ece(fixed, test_y):.3f}")

plt.plot([0,1],[0,1],"--",color="#888")
for conf, lab, c in [(raw,"before","#c44e52"), (fixed,"after","#55a467")]:
    cf, ac, _ = reliability_curve(conf, test_y, n_bins=12)
    plt.plot(cf, ac, "o-", color=c, label=lab)
plt.xlabel("confidence"); plt.ylabel("accuracy"); plt.legend()
plt.gca().set_aspect("equal"); plt.grid(alpha=0.3); plt.show()

This is the **same** temperature operation (logits / T) as decoding in Post 3a.
There it controlled sampling diversity; here it controls confidence honesty.

## 3. Calibration ≠ discrimination

Temperature scaling is monotonic, so it never changes the *ranking* of items
by confidence. The risk–coverage curve (accuracy on the most-confident k%) is
identical before and after — even though ECE collapsed.

In [ ]:
def risk_coverage(conf, y, n=30):
    order = np.argsort(-conf); y = np.asarray(y, float)[order]
    covs = np.linspace(0.05, 1.0, n)
    return covs, [y[:max(1,int(c*len(y)))].mean() for c in covs]

cov, acc_raw = risk_coverage(raw, test_y)
_,   acc_fix = risk_coverage(fixed, test_y)
plt.plot(cov, acc_raw, "o-", color="#c44e52", label="before")
plt.plot(cov, acc_fix, "s", color="#55a467", fillstyle="none", label="after")
plt.xlabel("coverage"); plt.ylabel("accuracy on answered"); plt.legend()
plt.grid(alpha=0.3); plt.gca().set_aspect("auto"); plt.show()
print("identical ranking:", np.allclose(acc_raw, acc_fix))

Calibration fixes *what the numbers mean*, not *how well they separate* right
from wrong. An agent needs both: discrimination from a capable model,
calibration from honest probabilities.

## 4. The agentic payoff

Gate on a fixed, face-value threshold: keep the model's answer if confidence
≥ τ, else fall back (retrieve / reflect / defer) with accuracy `fb`. A
calibrated threshold means what it says; an overconfident one keeps wrong
answers it is falsely sure of.

In [ ]:
fb, tau = 0.88, 0.8
thr = np.linspace(0.5, 0.98, 40)
acc_raw = [gated_decision_accuracy(raw, test_y, t, fb)[0] for t in thr]
acc_fix = [gated_decision_accuracy(fixed, test_y, t, fb)[0] for t in thr]
plt.plot(thr, acc_raw, "o-", color="#c44e52", label="overconfident")
plt.plot(thr, acc_fix, "s-", color="#55a467", label="temperature-scaled")
plt.axvline(0.8, color="#3a7ebf", ls=":", label="face-value 0.80")
plt.xlabel("threshold"); plt.ylabel("gated accuracy"); plt.legend()
plt.grid(alpha=0.3); plt.gca().set_aspect("auto"); plt.show()

i = int(np.argmin(np.abs(thr-0.8)))
print(f"at tau=0.80: raw={acc_raw[i]:.3f}, calibrated={acc_fix[i]:.3f}")

## What's next

You've seen calibration end to end:

- **Reliability diagram + ECE** — does confidence p mean accuracy p?
- **Overconfidence** — the default for modern nets, worsened by RLHF.
- **Temperature scaling** — one parameter, fit post-hoc, fixes most of it;
  the same knob as decoding temperature.
- **Calibration ≠ discrimination** — recalibration fixes the levels, not the
  ranking.
- **The agentic payoff** — a calibrated threshold is what makes "retrieve /
  reflect / act when unsure" behave as designed.

This closes **Topic 4** (memory, retrieval, reflection, uncertainty). Next,
**Topic 5** assembles every piece of the curriculum into complete agents:
multi-agent systems, then a from-scratch nanoAgent reference implementation.